# Revision experiments: regret vs disappointment RAG (local run)

This notebook runs **on your own computer**: no Colab, no Google Drive. It re-runs and extends the study
to answer the four reviewers. All logic lives in `../code/`; the notebook only drives it, so every number in
the revised paper traces back to a module and an output file under `../results/`.

**Before you start**
* Open this notebook from `Behavioral_Economics/Revision/notebooks/` in Jupyter / VS Code and run the cells in order
  (the second cell installs any missing packages automatically).
* `Behavioral_Economics/.env` must contain `OPENAI_API_KEY=...` (it is read automatically and never printed).
* The literature PDFs are read from `Behavioral_Economics/Materials/`. All four are indexed for the revised
  system; the *Mental Accounting* PDF alone is used for the replication of the original pipeline.

| Stage | What it does | API? | Approx. time | Answers |
|---|---|---|---|---|
| 0 | Setup: paths, packages, key | – | 1–5 min (first install) | – |
| 1 | Data preparation & description | no | < 1 min | R1.1, R4.4 |
| 2 | Re-analysis from reported statistics | no | < 1 min | R2.5, R3.4, R2.4/R3.3 |
| 3 | Cost estimate | no | seconds | – |
| 4 | Knowledge base | embeddings | 1–2 min | R4.2 |
| 5 | Ablations C0–C3, Speculative RAG C4, original replication R0 | **yes** | ~1–2 h total | R1.2, R2.1, R3.1, R3.5, R4.5–7 |
| 6 | Stability reruns, RDS, counterfactual focus, external set | **yes** | ~30–45 min | R2.4, R2.8, R4.9 |
| 7 | Baselines: lexicon, VADER (7a) · NLI, RoBERTa (7b, optional) | no | 7a < 1 min · 7b 30–90 min on CPU | R2.1, R4.5, R4.6 |
| 8 | Human annotation agreement → gold labels | no | seconds | R2.3, R3.2, R4.3 |
| 9 | Analysis + figures | no | < 1 min | all |

Every API call is cached in `results/llm_cache/`. If the kernel stops or you close the laptop, re-run the same cell:
finished reviews are not paid for twice. Times depend on your OpenAI rate limit (`MAX_WORKERS` below).

## 0 · Setup

In [ ]:
import os, sys
from pathlib import Path

def find_revision_dir() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, here.parent, here / "Revision", here.parent / "Revision"]:
        if (cand / "code" / "config.py").exists():
            return cand
    raise FileNotFoundError("Open this notebook from Behavioral_Economics/Revision/notebooks/ "
                            "or set REVISION_DIR manually below.")

REVISION_DIR = find_revision_dir()          # or: Path(r"C:\Users\...\Behavioral_Economics\Revision")
PROJECT_DIR = REVISION_DIR.parent
os.environ["REVISION_DIR"] = str(REVISION_DIR)
os.environ.setdefault("DATASET_DIR", str(PROJECT_DIR / "Dataset"))
os.environ.setdefault("MAX_WORKERS", "8")   # parallel API calls; lower to 2-4 if you hit rate limits
if str(REVISION_DIR / "code") not in sys.path:
    sys.path.insert(0, str(REVISION_DIR / "code"))
print("Project :", PROJECT_DIR)
print("Revision:", REVISION_DIR)
print("Python  :", sys.version.split()[0])

In [ ]:
# Installs any missing packages into THIS kernel's Python (works for uv, venv and conda kernels).
# Safe to re-run: does nothing when everything is already installed.
import importlib.util, shutil, subprocess

INSTALL_TRANSFORMERS = False             # optional stage 7b (torch + transformers, ~1-2 GB)
CORE = {"openai": "openai", "pandas": "pandas", "numpy": "numpy", "scipy": "scipy", "sklearn": "scikit-learn",
        "statsmodels": "statsmodels", "faiss": "faiss-cpu", "pypdf": "pypdf",
        "langchain_text_splitters": "langchain-text-splitters", "openpyxl": "openpyxl",
        "vaderSentiment": "vaderSentiment", "krippendorff": "krippendorff", "lingua": "lingua-language-detector",
        "matplotlib": "matplotlib", "tqdm": "tqdm", "ipywidgets": "ipywidgets"}

def _install(req_file):
    py = sys.executable
    uv = shutil.which("uv") or shutil.which("uv.exe")
    if uv:
        cmd = [uv, "pip", "install", "--python", py, "-r", str(req_file)]
    else:
        if importlib.util.find_spec("pip") is None:            # uv-created venvs ship without pip
            subprocess.run([py, "-m", "ensurepip", "--upgrade"], check=True)
        cmd = [py, "-m", "pip", "install", "-r", str(req_file)]
    print("Running:", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print((r.stdout or "")[-2500:], (r.stderr or "")[-2500:])
    r.check_returncode()

missing = [pkg for mod, pkg in CORE.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Missing packages:", missing)
    _install(REVISION_DIR / "requirements.txt")
    importlib.invalidate_caches()
else:
    print("All core packages already installed.")
if INSTALL_TRANSFORMERS and importlib.util.find_spec("transformers") is None:
    _install(REVISION_DIR / "requirements-transformers.txt")
still = [pkg for mod, pkg in CORE.items() if importlib.util.find_spec(mod) is None]
print("Still missing:", still or "none - continue with the next cell")

In [ ]:
import importlib, json
import pandas as pd
import config as C                       # importing config also loads ../.env
C.ensure_dirs()
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 200)

MOCK = os.environ.get("MOCK_LLM") == "1"     # code testing only (fake model) - never for results
RUN_LLM = True                                # False = run only the free stages (1-3, 7a, 8, 9)
RUN_TRANSFORMER_BASELINES = False             # stage 7b; needs requirements-transformers.txt

if RUN_LLM and not MOCK and not os.environ.get("OPENAI_API_KEY"):
    from getpass import getpass
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY not found in .env - paste it here: ")
print(".env variables loaded :", C.ENV_LOADED)
print("OpenAI key available  :", bool(os.environ.get("OPENAI_API_KEY")))
print("Models                :", C.MAIN_MODEL, "| drafter", C.DRAFTER_MODEL, "| embeddings", C.EMBED_MODEL)
print("Conditions            :", {k: v["enabled"] for k, v in C.CONDITIONS.items()})

In [ ]:
# Connectivity check (one cheap request, no tokens generated)
if RUN_LLM and not MOCK:
    from openai import OpenAI
    try:
        models = [m.id for m in OpenAI().models.list().data]
        print("OpenAI reachable. gpt-4o:", C.MAIN_MODEL in models, "| gpt-4o-mini:", C.DRAFTER_MODEL in models)
    except Exception as e:
        print("OpenAI NOT reachable:", repr(e)[:300])

## 1 · Data preparation (R1.1, R4.4)
Replicates the original preprocessing exactly, rebuilds the paper's 100-review sample (`seed = 42`), and draws a
500-review external sample from a different product line (Kindle/Fire/Echo, `Dataset/1429_1.csv`).

In [ ]:
import data_prep
summary = data_prep.run()
display(pd.read_csv(C.RESULTS_DIR / "01_data" / "data_flow.csv"))
display(pd.read_csv(C.RESULTS_DIR / "01_data" / "pool_vs_sample.csv", header=None, names=["statistic", "pool", "paper sample"]).head(16))
display(pd.read_csv(C.RESULTS_DIR / "01_data" / "category_pool_vs_sample.csv"))

## 2 · Re-analysis from the reported statistics (R2.5, R3.4, R2.4, R3.3)
Uses only the numbers printed in the manuscript: Welch's test, CIs, a CI for Cohen's d, power, an
internal-consistency check, and the algebra of the RDS formula.

In [ ]:
import reported_stats
rs = reported_stats.run()
display(pd.read_csv(C.RESULTS_DIR / "00_reported_stats" / "reanalysis_from_reported_stats.csv"))
print(json.dumps(rs["reanalysis"]["internal_consistency"], indent=2))
print(json.dumps(rs["algebra"], indent=2))
display(pd.read_csv(C.RESULTS_DIR / "00_reported_stats" / "power_table.csv"))

## 3 · Cost estimate (no API calls)

In [ ]:
import pipeline
est = pipeline.estimate()
display(est); print("Indicative total USD:", round(est.usd_indicative.sum(), 2), "(check current OpenAI pricing)")

## 4 · Knowledge base (R4.2)
Revised KB: every PDF in `Revision/knowledge_base/` if you put any there, otherwise every PDF in `Materials/`.
Original KB (replication R0): the *Mental Accounting* PDF, which was the only file the original notebook indexed.

In [ ]:
print("Revised KB :", [p.name for p in C.kb_pdfs()])
print("Original KB:", [p.name for p in C.original_kb_pdfs()])
assert C.kb_pdfs(), "No PDFs found in Materials/ or knowledge_base/"

In [ ]:
runner = pipeline.Runner(mock=MOCK)
runner.stage_kb()
display(pd.read_csv(C.RESULTS_DIR / "02_knowledge_base" / "kb_revised_documents.csv"))

## 5 · Classification: ablation grid, Speculative RAG, original replication
* **C0** labels only · **C1** + definitions · **C2** retrieval only · **C3** retrieval + definitions (proposed) · **C4** Speculative RAG
* **R0** the paper's exact prompt with the original single-PDF knowledge base

Each cell can be interrupted and re-run; cached reviews are skipped.

In [ ]:
runner.stage_classify()                 # C0-C3 on 2,019 reviews (~8k calls)
display(pd.read_csv(C.RESULTS_DIR / "02_knowledge_base" / "retrieval_sources_revised.csv"))

In [ ]:
runner.stage_speculative()              # C4: 4 drafter + 8 verifier calls per review (~24k calls)

In [ ]:
runner.stage_original()                 # R0: original prompt + original KB

## 6 · Stability reruns, RDS dimensions, counterfactual focus, external dataset

In [ ]:
runner.stage_stability()
runner.stage_rds()
runner.stage_cf()
runner.stage_external()
master = runner.build_master(); runner.save_usage()
master.filter(like="__label").apply(pd.Series.value_counts).T

## 7 · Conventional baselines (R2.1, R4.5, R4.6)
**7a** (lexicon, VADER) is instant. **7b** (zero-shot NLI with BART-large-MNLI, RoBERTa) is optional: it needs
`requirements-transformers.txt`, downloads models from Hugging Face on first use, and runs on CPU
(about 30–90 min; much faster with an NVIDIA GPU). RoBERTa models train only once gold labels exist (stage 8).

In [ ]:
import baselines_classic as BC
for ds in ["pool", "external"]:
    BC.run_pool(ds)
display(pd.read_csv(C.RESULTS_DIR / "05_baselines" / "pool_lexicon_vader_summary.csv"))

In [ ]:
if RUN_TRANSFORMER_BASELINES and not MOCK:
    import baselines_transformer as BT
    print(BT.run())                     # NLI on the pool; RoBERTa CV + silver once gold labels exist
else:
    print("Stage 7b skipped (set RUN_TRANSFORMER_BASELINES = True in the setup cell)")

## 8 · Human annotation → gold labels (R2.3, R3.2, R4.3)
1. Two coders independently fill `annotation/annotation_coder_A.xlsx` and `annotation_coder_B.xlsx` (codebook inside).
2. Save the completed files into `annotation/returned/` under the same names and run the next cell.
3. Resolve disagreements together in `annotation/adjudication.xlsx`, save it into `annotation/returned/`, and run the cell again.

In [ ]:
import annotation
annotation.build_sheets()               # no-op if the sheets already exist (never overwrites)
res = annotation.agreement()
print(res)
f = C.RESULTS_DIR / "04_human_annotation" / "inter_rater_agreement.csv"
if f.exists(): display(pd.read_csv(f))

In [ ]:
# Supervised baselines that need gold labels
gf = C.DATA_DIR / "gold_labels.csv"
if gf.exists():
    gold = pd.read_csv(gf).merge(pd.read_csv(C.DATA_DIR / "pool.csv")[["review_uid", "reviews.text"]], on="review_uid")
    BC.tfidf_cv(gold)
    if RUN_TRANSFORMER_BASELINES and not MOCK:
        import baselines_transformer as BT
        print(BT.run(skip_nli=True))    # RoBERTa CV + silver
else:
    print("gold_labels.csv not available yet")

## 9 · Analysis and figures

In [ ]:
import analysis, figures
importlib.reload(analysis); importlib.reload(figures)
analysis.run(); figures.run()
print((C.RESULTS_DIR / "06_analysis" / "STATUS.md").read_text(encoding="utf-8"))

In [ ]:
A = C.RESULTS_DIR / "06_analysis"
for f in ["A_prevalence_by_system.csv", "A_label_changes_per_component.csv", "B_accuracy_vs_human.csv",
          "B_mcnemar_ablation_tests.csv", "C_test_retest_stability.csv", "D_rating_correlations.csv",
          "D_weight_sensitivity.csv", "G_generalisability.csv"]:
    if (A / f).exists():
        print("\n###", f); display(pd.read_csv(A / f))
for f in ["D_rds_group_comparisons.csv"]:
    if (A / f).exists():
        print("\n###", f); display(pd.read_csv(A / f, index_col=0))

In [ ]:
from IPython.display import Image, display as show
for p in sorted((C.RESULTS_DIR / "07_figures").glob("*.png")):
    print(p.name); show(Image(filename=str(p), width=720))